## transition of committment portfolio of companies in global fortune 500 from 2021 to 2025

CN nz transitions 

In [6]:
import pandas as pd
import plotly.graph_objects as go

years = ['2021', '2022', '2023', '2024']
dfs = {year: pd.read_excel("historic new .xlsx", sheet_name=year) for year in years}

for year in years:
    dfs[year].columns = dfs[year].columns.str.strip().str.replace('\ufeff', '')

def get_state(row):
    has_cn = pd.notna(row.get('CN')) and row['CN'] != 'None'
    has_nz = pd.notna(row.get('NZ')) and row['NZ']  != 'None'
    
    if has_cn and has_nz:
        return 'CN+NZ'
    elif has_cn:
        return 'CN only'
    elif has_nz:
        return 'NZ only'
    return 'None'

states = []
for year in years:
    for _, row in dfs[year].iterrows():
        states.append({
            'company': row['company'],
            'year': year,
            'state': get_state(row)
        })

state_df = pd.DataFrame(states).pivot(index='company', columns='year', values='state')

flows = []
for i in range(len(years)-1):
    curr_year = years[i]
    next_year = years[i+1]
    
    for company in state_df.index:
        curr_state = state_df.loc[company, curr_year]
        next_state = state_df.loc[company, next_year]
        
        source = f"{curr_year}_{curr_state if pd.notna(curr_state) else 'NEW'}"
        target = f"{next_year}_{next_state if pd.notna(next_state) else 'LEFT'}"
        
        flows.append({'source': source, 'target': target, 'value': 1})

flow_df = pd.DataFrame(flows).groupby(['source', 'target']).sum().reset_index()

all_nodes = sorted(set(flow_df['source'].tolist() + flow_df['target'].tolist()))
node_dict = {node: idx for idx, node in enumerate(all_nodes)}

colors = {
    'CN+NZ': '#8B479B',
    'CN only': '#E84E8A', 
    'NZ only': '#00B4B2',
    'None': '#858282ff',
    'NEW': '#90EE90',
    'LEFT': '#FFB6C1'
}

node_colors = [colors.get(node.split('_')[1], '#D3D3D3') for node in all_nodes]

flow_df['source_idx'] = flow_df['source'].map(node_dict)
flow_df['target_idx'] = flow_df['target'].map(node_dict)

fig = go.Figure(data=[go.Sankey(
    node=dict(label=all_nodes, color=node_colors, pad=20, thickness=20),
    link=dict(
        source=flow_df['source_idx'],
        target=flow_df['target_idx'],
        value=flow_df['value']
    )
)])

fig.update_layout(title="CN & NZ Commitment Transitions 2021-2024", height=800)
fig.write_html('/mnt/user-data/outputs/cn_nz_sankey.html')

ValueError: Index contains duplicate entries, cannot reshape

sbti nt nz transitions 